In [3]:
from datasets import load_dataset
import os

dataset = load_dataset("mattymchen/celeba-hq", split="train[:5000]")

save_dir = "celeba_hq_5000"
os.makedirs(save_dir, exist_ok=True)

for i, example in enumerate(dataset):
    example['image'].save(os.path.join(save_dir, f"{i:05d}.jpg"))
    if (i + 1) % 500 == 0:
        print(f"save {i+1} images")


Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 4596.35 examples/s]


save 500 images
save 1000 images
save 1500 images
save 2000 images
save 2500 images
save 3000 images
save 3500 images
save 4000 images
save 4500 images
save 5000 images


In [1]:
import os
from datasets import load_dataset
import pandas as pd
import requests

# ── 1. 下载官方映射文件 ──────────────────────────────────────────
# 来源: https://drive.google.com/file/d/1badu11NqxGf6qM3PTTooiDoCFBEvGDUv
# 如果已经手动下载好，直接指定本地路径
MAPPING_PATH = "CelebA-HQ-to-CelebA-mapping.txt"

# 映射文件格式（空格分隔，第一行是header）:
# idx  orig_idx  orig_file
# 0    6097      006098.jpg
# 1    8409      008410.jpg
# ...

mapping_df = pd.read_csv(
    MAPPING_PATH,
    sep=r'\s+',
    header=0,         # 第一行是列名
    dtype={'idx': int, 'orig_idx': int, 'orig_file': str}
)

# ── 2. 判断每个 HQ 图片是否属于 CelebA 测试集 ───────────────────
# 你的 split 规则:
#   train : iloc[:162770]   → orig_idx 0~162769   (1-based: 000001~162770)
#   val   : iloc[162770:182637]
#   test  : iloc[182637:]   → orig_idx >= 182637  (1-based: 182638~202599)

TEST_START = 182637   # 对应 iloc 索引，即 0-based 的 182637

# orig_idx 在映射文件里是 0-based（与 df.iloc 一致）
test_mask = mapping_df['orig_idx'] >= TEST_START
test_hq_indices = mapping_df[test_mask]['idx'].tolist()  # celeba-hq 里的 index

print(f"CelebA-HQ 30000 张中属于 CelebA 测试集的数量: {len(test_hq_indices)}")
print(f"前10个 HQ index: {test_hq_indices[:10]}")

# ── 3. 加载 celeba-hq，只保存属于测试集的图片 ────────────────────
dataset = load_dataset("mattymchen/celeba-hq", split="train")  # 加载全部30000张

test_hq_set = set(test_hq_indices)

save_dir = "celeba_hq_test"
os.makedirs(save_dir, exist_ok=True)

saved = 0
limit = 5000
for i, example in enumerate(dataset):
    if i in test_hq_set:
        example['image'].save(os.path.join(save_dir, f"{i:05d}.jpg"))
        saved += 1
        if saved % 200 == 0:
            print(f"已保存 {saved} 张")
        if saved >= limit:
            break
print(f"完成，共保存 {saved} 张测试集图片 → {save_dir}/")

/home/chen/miniconda3/envs/DETECT/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CelebA-HQ 30000 张中属于 CelebA 测试集的数量: 2824
前10个 HQ index: [2, 4, 26, 44, 59, 84, 93, 108, 127, 133]
已保存 200 张
已保存 400 张
已保存 600 张
已保存 800 张
已保存 1000 张
已保存 1200 张
已保存 1400 张
已保存 1600 张
已保存 1800 张
已保存 2000 张
已保存 2200 张
已保存 2400 张
已保存 2600 张
完成，共保存 2656 张测试集图片 → celeba_hq_test/


In [1]:
#
import os
import sys
import zipfile
import argparse
from pathlib import Path

SUPPORTED_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def folder_to_zip(input_dir: str, output_zip: str, verbose: bool = True):
    input_path = Path(input_dir)
    if not input_path.exists():
        print(f"[ERROR] 输入目录不存在: {input_dir}")
        sys.exit(1)

    # 收集所有图片（递归）
    all_images = sorted([
        p for p in input_path.rglob('*')
        if p.suffix.lower() in SUPPORTED_EXT
    ])

    if len(all_images) == 0:
        print(f"[ERROR] 未找到任何图片文件（支持格式: {SUPPORTED_EXT}）")
        sys.exit(1)

    print(f"找到 {len(all_images)} 张图片，开始打包...")

    output_path = Path(output_zip)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(output_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        for i, img_path in enumerate(all_images):
            # StyleGAN 要求：图片直接放在 zip 根目录，不带子文件夹路径
            arcname = img_path.name
            zf.write(img_path, arcname=arcname)

            if verbose and (i + 1) % 500 == 0:
                print(f"  已打包 {i + 1} / {len(all_images)}")

    size_mb = output_path.stat().st_size / (1024 ** 2)
    print(f"\n完成！")
    print(f"  输出文件: {output_path.resolve()}")
    print(f"  图片数量: {len(all_images)}")
    print(f"  文件大小: {size_mb:.1f} MB")

folder_to_zip("celeba_hq_test", "celeba_hq_test.zip", verbose=True)

找到 2656 张图片，开始打包...
  已打包 500 / 2656
  已打包 1000 / 2656
  已打包 1500 / 2656
  已打包 2000 / 2656
  已打包 2500 / 2656

完成！
  输出文件: /data/chen/detect/stylegan3/celeba_hq_test.zip
  图片数量: 2656
  文件大小: 233.4 MB


In [3]:
import os
from PIL import Image
from collections import Counter

def check_resolutions(image_dir):
    resolutions = []
    issues = []

    print(f"checking image folder: {image_dir}")
    files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for filename in files:
        path = os.path.join(image_dir, filename)
        try:
            with Image.open(path) as img:
                width, height = img.size
                resolutions.append((width, height))

                # StyleGAN2 要求必须是正方形
                if width != height:
                    issues.append(f"not square: {filename} ({width}x{height})")
        except Exception as e:
            issues.append(f"unable to read: {filename} - {e}")

    # 统计分布
    stats = Counter(resolutions)

    print("\n--- result ---")
    for res, count in stats.items():
        print(f"resolution {res[0]}x{res[1]}: {count} images")

    if issues:
        print("\n--- issues---")
        for issue in issues[:10]: # 只列出前10个
            print(f"[!] {issue}")
        if len(issues) > 10:
            print(f"... other {len(issues)-10} issues")

# 使用方法
check_resolutions('./celeba_hq_5000')

checking image folder: ./celeba_hq_5000

--- result ---
resolution 1024x1024: 5000 images


In [4]:
!python dataset_tool.py --source=./celeba_hq_5000 --dest=./dataset/celeba.zip
!python train.py --outdir=./training-runs --data=./dataset/celeba.zip \
    --gpus=1 --cfg=paper1024 --resume=../local_models/generators/stylegan2-ffhq-1024x1024.pkl --snap=10 \
    --aug=ada --target=0.6 \
    --freezed=10

Traceback (most recent call last):
  File "/data/chen/detect/stylegan2-ada-pytorch-main/dataset_tool.py", line 24, in <module>
    from tqdm import tqdm
ModuleNotFoundError: No module named 'tqdm'
